### Q1. Handle Missing Values (Basic Cleaning) — Fill missing values: Age → Median, Salary → Mean, Hours_Worked_Per_Week → Median, Performance_Score → Mean. Display the dataset after handling missing values.

-->  

Answer:  
Before cleaning, the dataset had missing values in four columns: Age (25 missing), Hours_Worked_Per_Week (50 missing), Performance_Score (50 missing), and Salary (25 missing). As instructed, Age and Hours_Worked_Per_Week (which can have skew from junior vs senior employees) were filled using median, while Salary and Performance_Score (more evenly distributed numeric fields) were filled using mean. The median Age used was 29.0, mean Salary was ≈₹50,666.67, median Hours_Worked_Per_Week was 41.0, and mean Performance_Score was ≈4.31. After this step, all four columns show zero missing values.

```python
import pandas as pd

df = pd.read_csv('employee_productivity_dataset.csv')

print("Missing values before cleaning:\n", df.isnull().sum())

df['Age'] = df['Age'].fillna(df['Age'].median())
df['Salary'] = df['Salary'].fillna(df['Salary'].mean())
df['Hours_Worked_Per_Week'] = df['Hours_Worked_Per_Week'].fillna(df['Hours_Worked_Per_Week'].median())
df['Performance_Score'] = df['Performance_Score'].fillna(df['Performance_Score'].mean())

print("\nMissing values after cleaning:\n", df.isnull().sum())
print("\nDataset after handling missing values:\n", df.head(10))
```

---
---

### Q2. Label Encoding — Convert Gender and Department into numeric values using Label Encoding. Show the updated columns.

-->  

Answer:  
Label Encoding was applied to Gender and Department since these are categorical columns suitable for ordinal-style numeric conversion (Gender has only 2 categories, and Department, while having 4 categories, is being encoded directly as instructed). Gender was mapped as Female → 0, Male → 1. Department was mapped as Finance → 0, HR → 1, IT → 2, Marketing → 3. Two new columns, Gender_Encoded and Department_Encoded, were added showing the numeric versions alongside the original text columns.

```python
from sklearn.preprocessing import LabelEncoder

le_gender = LabelEncoder()
le_dept = LabelEncoder()

df['Gender_Encoded'] = le_gender.fit_transform(df['Gender'])
df['Department_Encoded'] = le_dept.fit_transform(df['Department'])

print("Gender mapping:", dict(zip(le_gender.classes_, le_gender.transform(le_gender.classes_))))
print("Department mapping:", dict(zip(le_dept.classes_, le_dept.transform(le_dept.classes_))))

print(df[['Gender', 'Gender_Encoded', 'Department', 'Department_Encoded']].head(10))
```

---
---

### Q3. One-Hot Encoding — Apply One-Hot Encoding on Work_Mode and Location. Display the dataset and check how many new columns are created.

-->  

Answer:  
One-Hot Encoding was applied to Work_Mode (3 categories: Remote, Hybrid, Onsite) and Location (4 categories: Pune, Mumbai, Bangalore, Delhi), since these are nominal categories with no natural order — unlike Label Encoding, One-Hot avoids implying a false ranking between them. Using `pd.get_dummies()`, the original Work_Mode and Location columns were replaced with 7 new binary columns in total: 3 for Work_Mode (Work_Mode_Hybrid, Work_Mode_Onsite, Work_Mode_Remote) and 4 for Location (Location_Bangalore, Location_Delhi, Location_Mumbai, Location_Pune). The dataset grew from 14 columns to 19 columns, a net increase of 5 columns (since the original 2 columns were removed and 7 new ones were added).

```python
cols_before = df.shape[1]

df = pd.get_dummies(df, columns=['Work_Mode', 'Location'])

cols_after = df.shape[1]

print("Columns before One-Hot Encoding:", cols_before)
print("Columns after One-Hot Encoding:", cols_after)
print("New columns created:", cols_after - cols_before)

print(df.head(10))
```

---
---

### Q4. Normalization (Min-Max Scaling) — Normalize Salary and Hours_Worked_Per_Week using MinMaxScaler. Display results.

-->  

Answer:  
MinMaxScaler was applied to Salary and Hours_Worked_Per_Week to rescale both columns into a fixed range of 0 to 1. This is useful because Salary and Hours_Worked_Per_Week have very different original scales (Salary in tens of thousands, Hours in tens), and normalization brings them onto a comparable scale, which helps distance-based ML algorithms (like KNN or clustering) treat both features fairly. After transformation, the lowest value in each column became 0, the highest became 1, and everything else was scaled proportionally in between.

```python
from sklearn.preprocessing import MinMaxScaler

mm_scaler = MinMaxScaler()

df[['Salary_Normalized', 'Hours_Worked_Per_Week_Normalized']] = mm_scaler.fit_transform(
    df[['Salary', 'Hours_Worked_Per_Week']]
)

print(df[['Salary', 'Salary_Normalized', 'Hours_Worked_Per_Week', 'Hours_Worked_Per_Week_Normalized']].head(10))
```

---
---

### Q5. Standardization (Scaling) — Apply StandardScaler on Age and Projects_Completed. Display the transformed values.

-->  

Answer:  
StandardScaler was applied to Age and Projects_Completed, transforming both columns so that they have a mean of 0 and a standard deviation of 1 (verified: mean after scaling ≈ 0, std ≈ 1.0). Unlike Min-Max scaling, standardization doesn't bound values to a fixed range — it centers the data around the mean and expresses each value in terms of how many standard deviations it is from that mean. This is preferred for algorithms that assume normally distributed data (like Logistic Regression or SVM), and is more robust when there isn't a fixed natural minimum/maximum like with Age.

```python
from sklearn.preprocessing import StandardScaler

std_scaler = StandardScaler()

df[['Age_Standardized', 'Projects_Completed_Standardized']] = std_scaler.fit_transform(
    df[['Age', 'Projects_Completed']]
)

print(df[['Age', 'Age_Standardized', 'Projects_Completed', 'Projects_Completed_Standardized']].head(10))
```

---
---

### Q6. Compare Scaling Methods — Apply both MinMaxScaler and StandardScaler on the Salary column. Show both results side by side.

-->  

Answer:  
Both scaling methods were applied to the Salary column to compare their effect. MinMaxScaler rescales Salary into a fixed 0–1 range, where the lowest salary becomes 0.0 and the highest becomes 1.0. StandardScaler instead centers Salary around a mean of 0 with a standard deviation of 1, so values can be negative (below average) or positive (above average), rather than being bounded.

Looking at the same employee, e.g. a salary of ₹50,000 becomes 0.348 under MinMax (just below the midpoint of the range) but becomes -0.097 under Standard scaling (just slightly below the mean). This shows the two methods produce very different transformed values even from the same input — MinMax preserves relative position within the range, while Standard expresses position relative to the mean and spread.

```python
from sklearn.preprocessing import MinMaxScaler, StandardScaler

mm = MinMaxScaler()
ss = StandardScaler()

df['Salary_MinMax'] = mm.fit_transform(df[['Salary']])
df['Salary_Standard'] = ss.fit_transform(df[['Salary']])

print(df[['Salary', 'Salary_MinMax', 'Salary_Standard']].head(10))
```

---
---

### Q7. Build Preprocessing Pipeline — Create a pipeline that applies encoding to categorical columns and scaling to numerical columns, using ColumnTransformer and Pipeline.

-->  

Answer:  
A reusable preprocessing pipeline was built using ColumnTransformer to apply different transformations to different column types in a single step. The categorical columns (Gender, Department, Work_Mode, Location) are passed through OneHotEncoder, while the numerical columns (Age, Salary, Hours_Worked_Per_Week, Projects_Completed, Performance_Score) are passed through StandardScaler. This is wrapped inside a Pipeline object, which makes the entire preprocessing sequence reusable and consistent — useful for applying the exact same transformations later on new/test data without repeating code.

```python
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

categorical_cols = ['Gender', 'Department', 'Work_Mode', 'Location']
numerical_cols = ['Age', 'Salary', 'Hours_Worked_Per_Week', 'Projects_Completed', 'Performance_Score']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(), categorical_cols)
])

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor)
])

print("Pipeline created successfully:")
print(pipeline)
```

---
---

### Q8. Apply Pipeline — Apply the pipeline on the dataset. Display the transformed dataset and shape of final dataset.

-->  

Answer:  
The pipeline built in Q7 was fitted and applied to the dataset using `fit_transform()`. The original 9 relevant columns (5 numerical + 4 categorical) expanded to 18 final columns after transformation — the 5 numerical columns stayed as 5 (just scaled), while the 4 categorical columns expanded into 13 one-hot encoded binary columns (2 for Gender, 4 for Department, 3 for Work_Mode, 4 for Location). The final transformed dataset shape is (250, 18) — same number of rows (250 employees), but more columns due to one-hot expansion.

```python
transformed = pipeline.fit_transform(df)

feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out()
transformed_df = pd.DataFrame(transformed, columns=feature_names)

print("Transformed Dataset:\n", transformed_df.head(10))
print("\nShape of final dataset:", transformed_df.shape)
```

---
---

### Q9. Conceptual Question — Why is scaling important in Python (Machine Learning)?

-->  

Answer:  
Scaling is important because many machine learning algorithms calculate distances or gradients based on the magnitude of feature values. If features are on very different scales — for example, Salary in tens of thousands versus Age in tens — the algorithm may give disproportionate importance to the feature with the larger numeric range, even if it isn't actually more important.

Algorithms like KNN, K-Means, SVM, Logistic Regression, and gradient-descent–based models are especially sensitive to this, since unscaled features can slow down convergence or bias the model's decision boundary. Scaling (via MinMaxScaler or StandardScaler) brings all numerical features to a comparable range, ensuring the model treats each feature fairly based on its actual relationship with the target, not its raw scale.

---
---

### Q10. Conceptual Question — Why do we convert categorical data into numerical form?

-->  

Answer:  
Machine learning algorithms are fundamentally mathematical — they work with numbers, performing calculations like distances, dot products, and gradients. Categorical data, such as "Male"/"Female" or "IT"/"HR"/"Finance", is text-based and cannot be directly processed by these mathematical operations.

Converting categorical data into numerical form (using Label Encoding or One-Hot Encoding) allows the model to interpret and use this information meaningfully. The choice of method matters too: Label Encoding is suitable for ordinal data or binary categories, while One-Hot Encoding is preferred for nominal categories with no inherent order, since it avoids falsely implying a ranking between categories (e.g., implying "Finance" < "HR" < "IT" numerically, which has no real meaning).

---
---